# AI-Powered Power BI Semantic Model Assistant
## Microsoft Fabric | Natural Language to DAX | Production-Ready

This notebook implements a complete AI assistant that understands Power BI Semantic Models and generates correct DAX expressions from natural language queries.

### Core Capabilities:
- 🧠 **Dynamic Model Understanding**: Automatically extracts metadata from Spark tables
- ⚡ **DAX Generation**: Converts natural language to valid DAX expressions using OpenAI
- ✅ **Smart Validation**: Checks schema, prevents errors, avoids duplicates
- 📚 **Self-Documenting**: Explains generated code in plain English
- 🎯 **Enterprise-Ready**: Modular, extensible design for real-world use

---

## Section 1: Environment Setup & Dependencies
Install and configure required libraries, OpenAI API, and logging for the assistant.

In [ ]:
# Install required packages
import subprocess
import sys

def install_packages():
    """Install required packages if not present"""
    packages = [
        'openai>=1.0.0',
        'pandas>=1.5.0',
        'pydantic>=2.0.0'
    ]
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        except Exception as e:
            print(f"Note: {package} installation skipped - {str(e)}")

print("Installing dependencies...")
install_packages()
print("✓ Dependencies installation complete")

In [ ]:
# Import all required libraries
import os
import json
import logging
import re
from typing import Dict, List, Optional, Tuple, Any
from datetime import datetime
from collections import defaultdict
import pandas as pd
from pydantic import BaseModel, Field

# OpenAI imports
from openai import OpenAI
os.environ['OPENAI_API_KEY'] = 'sk-proj-***REMOVED***'  # Add your key here
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - [%(name)s] - %(message)s'
)
logger = logging.getLogger(__name__)

# Configure OpenAI API
def configure_openai_api():
    """Configure OpenAI API key from environment"""
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        print("\n⚠️  Warning: OPENAI_API_KEY not found in environment")
        print("   Please set your API key: export OPENAI_API_KEY='your-key'")
        print("   Or enter it directly below:")
        api_key = input("Enter your OpenAI API key: ").strip()
    
    if api_key:
        os.environ['OPENAI_API_KEY'] = api_key
        client = OpenAI(api_key=api_key)
        logger.info("✓ OpenAI API configured successfully")
        return client
    else:
        logger.warning("⚠️  No API key provided. Some features will be limited.")
        return None

# Initialize OpenAI client
openai_client = configure_openai_api()

print("✓ Environment setup complete")

## Section 2: Spark Data Loader
Load Delta tables from Lakehouse and extract structural metadata (tables, columns, data types).

In [ ]:
class SparkDataLoader:
    """Load and extract metadata from Spark/Delta tables"""
    
    def __init__(self):
        """Initialize Spark session if available"""
        self.spark = None
        self.tables_cache = {}
        self._init_spark()
    
    def _init_spark(self):
        """Initialize Spark session (works in Fabric notebooks)"""
        try:
            # In Fabric notebooks, spark is already available
            self.spark = spark  # This assumes we're in a Fabric environment
            logger.info("✓ Spark session initialized")
        except NameError:
            # For local testing without Fabric
            from pyspark.sql import SparkSession
            try:
                self.spark = SparkSession.builder.appName("PowerBI_Assistant").getOrCreate()
                logger.info("✓ Spark session created locally")
            except:
                logger.warning("⚠️  Could not initialize Spark session - offline mode")
    
    def get_available_tables(self) -> List[str]:
        """Get list of all available tables in the Lakehouse"""
        if not self.spark:
            logger.warning("Spark not available. Using sample tables.")
            return ["Sales", "Product", "Region", "Salesperson", "SalespersonRegion", "Targets"]
        
        try:
            tables = self.spark.catalog.listTables()
            table_names = [t.name for t in tables]
            logger.info(f"✓ Found {len(table_names)} tables: {table_names}")
            return table_names
        except Exception as e:
            logger.warning(f"Could not list tables: {e}. Using sample tables.")
            return ["Sales", "Product", "Region", "Salesperson", "SalespersonRegion", "Targets"]
    
    def get_table_schema(self, table_name: str) -> Dict[str, str]:
        """
        Extract column names and data types from a table
        Returns: {column_name: data_type}
        """
        if not self.spark:
            return self._get_sample_schema(table_name)
        
        try:
            df = self.spark.table(table_name)
            schema = {field.name: str(field.dataType) for field in df.schema.fields}
            logger.info(f"✓ Schema for '{table_name}': {len(schema)} columns")
            self.tables_cache[table_name] = schema
            return schema
        except Exception as e:
            logger.warning(f"Could not load schema for {table_name}: {e}. Using sample schema.")
            return self._get_sample_schema(table_name)
    
    def _get_sample_schema(self, table_name: str) -> Dict[str, str]:
        """Provide sample schemas for demonstration"""
        schemas = {
            "Sales": {
                "SalesKey": "bigint",
                "ProductKey": "bigint",
                "EmployeeKey": "bigint",
                "RegionKey": "bigint",
                "OrderDate": "date",
                "Sales": "decimal(18,2)",
                "Quantity": "int",
                "OrderID": "string"
            },
            "Product": {
                "ProductKey": "bigint",
                "ProductName": "string",
                "Category": "string",
                "SubCategory": "string",
                "Price": "decimal(18,2)"
            },
            "Region": {
                "RegionKey": "bigint",
                "RegionName": "string",
                "Country": "string"
            },
            "Salesperson": {
                "EmployeeKey": "bigint",
                "EmployeeID": "string",
                "EmployeeName": "string",
                "Title": "string"
            },
            "SalespersonRegion": {
                "EmployeeKey": "bigint",
                "RegionKey": "bigint"
            },
            "Targets": {
                "EmployeeID": "string",
                "TargetAmount": "decimal(18,2)",
                "Year": "int"
            }
        }
        return schemas.get(table_name, {})
    
    def get_sample_data(self, table_name: str, limit: int = 5) -> Optional[pd.DataFrame]:
        """Get sample data from a table"""
        if not self.spark:
            return None
        
        try:
            df = self.spark.table(table_name).limit(limit).toPandas()
            logger.info(f"✓ Retrieved {len(df)} sample rows from {table_name}")
            return df
        except Exception as e:
            logger.warning(f"Could not retrieve sample data: {e}")
            return None

# Initialize loader
spark_loader = SparkDataLoader()
print("✓ Spark Data Loader initialized")

## Section 3: Metadata Extractor & Schema Builder
Build complete semantic model metadata including tables, columns, relationships, and measures.

In [ ]:
class SemanticModelMetadata:
    """Manage Power BI Semantic Model metadata"""
    
    def __init__(self, loader: SparkDataLoader):
        """Initialize metadata manager"""
        self.loader = loader
        self.metadata = {
            "tables": {},
            "relationships": [],
            "measures": {},
            "calculated_columns": {}
        }
        self._build_metadata()
    
    def _build_metadata(self):
        """Build complete semantic model metadata"""
        logger.info("Building semantic model metadata...")
        
        # Extract all tables and their schemas
        tables = self.loader.get_available_tables()
        for table_name in tables:
            schema = self.loader.get_table_schema(table_name)
            self.metadata["tables"][table_name] = {
                "columns": schema,
                "column_count": len(schema)
            }
        
        # Define explicit relationships (following Power BI conventions)
        self.metadata["relationships"] = [
            {
                "name": "Sales_Product",
                "from_table": "Sales",
                "from_column": "ProductKey",
                "to_table": "Product",
                "to_column": "ProductKey"
            },
            {
                "name": "Sales_Salesperson",
                "from_table": "Sales",
                "from_column": "EmployeeKey",
                "to_table": "Salesperson",
                "to_column": "EmployeeKey"
            },
            {
                "name": "Sales_Region",
                "from_table": "Sales",
                "from_column": "RegionKey",
                "to_table": "Region",
                "to_column": "RegionKey"
            },
            {
                "name": "SalespersonRegion_Salesperson",
                "from_table": "SalespersonRegion",
                "from_column": "EmployeeKey",
                "to_table": "Salesperson",
                "to_column": "EmployeeKey"
            },
            {
                "name": "SalespersonRegion_Region",
                "from_table": "SalespersonRegion",
                "from_column": "RegionKey",
                "to_table": "Region",
                "to_column": "RegionKey"
            },
            {
                "name": "Targets_Salesperson",
                "from_table": "Targets",
                "from_column": "EmployeeID",
                "to_table": "Salesperson",
                "to_column": "EmployeeID"
            }
        ]
        
        # Define existing measures (sample)
        self.metadata["measures"] = {
            "Total_Sales": {
                "expression": "SUM(Sales[Sales])",
                "description": "Total sum of all sales"
            },
            "Total_Quantity": {
                "expression": "SUM(Sales[Quantity])",
                "description": "Total quantity sold"
            },
            "Average_Price": {
                "expression": "AVERAGE(Product[Price])",
                "description": "Average product price"
            },
            "Total_Employees": {
                "expression": "DISTINCTCOUNT(Sales[EmployeeKey])",
                "description": "Count of unique sales employees"
            }
        }
        
        logger.info(f"✓ Metadata built: {len(self.metadata['tables'])} tables, "
                   f"{len(self.metadata['relationships'])} relationships, "
                   f"{len(self.metadata['measures'])} measures")
    
    def get_table_columns(self, table_name: str) -> List[str]:
        """Get column names for a table"""
        if table_name not in self.metadata["tables"]:
            return []
        return list(self.metadata["tables"][table_name]["columns"].keys())
    
    def get_column_type(self, table_name: str, column_name: str) -> Optional[str]:
        """Get data type of a column"""
        if table_name not in self.metadata["tables"]:
            return None
        return self.metadata["tables"][table_name]["columns"].get(column_name)
    
    def check_table_exists(self, table_name: str) -> bool:
        """Check if a table exists in the model"""
        return table_name in self.metadata["tables"]
    
    def check_column_exists(self, table_name: str, column_name: str) -> bool:
        """Check if a column exists in a table"""
        if not self.check_table_exists(table_name):
            return False
        return column_name in self.metadata["tables"][table_name]["columns"]
    
    def get_relationships_for_table(self, table_name: str) -> List[Dict]:
        """Get all relationships connected to a table"""
        related = []
        for rel in self.metadata["relationships"]:
            if rel["from_table"] == table_name or rel["to_table"] == table_name:
                related.append(rel)
        return related
    
    def get_related_table(self, from_table: str, foreign_key: str) -> Optional[Tuple[str, str]]:
        """Find related table through a foreign key"""
        for rel in self.metadata["relationships"]:
            if rel["from_table"] == from_table and rel["from_column"] == foreign_key:
                return rel["to_table"], rel["to_column"]
        return None
    
    def get_all_measures(self) -> Dict[str, str]:
        """Get all existing measures"""
        return {name: m["expression"] for name, m in self.metadata["measures"].items()}
    
    def add_measure(self, name: str, expression: str, description: str = ""):
        """Add a new measure to metadata"""
        self.metadata["measures"][name] = {
            "expression": expression,
            "description": description,
            "created_at": datetime.now().isoformat()
        }
        logger.info(f"✓ Measure added: {name}")
    
    def measure_exists(self, name: str) -> bool:
        """Check if a measure already exists"""
        return name in self.metadata["measures"]
    
    def to_json(self) -> str:
        """Export metadata as JSON"""
        return json.dumps(self.metadata, indent=2, default=str)
    
    def get_summary(self) -> str:
        """Get readable summary of the model"""
        summary = "📊 SEMANTIC MODEL SUMMARY\n"
        summary += "=" * 50 + "\n\n"
        
        summary += f"📋 TABLES ({len(self.metadata['tables'])})\n"
        for table_name, info in self.metadata["tables"].items():
            summary += f"  • {table_name}: {info['column_count']} columns\n"
        
        summary += f"\n🔗 RELATIONSHIPS ({len(self.metadata['relationships'])})\n"
        for rel in self.metadata["relationships"]:
            summary += f"  • {rel['from_table']}.{rel['from_column']} → {rel['to_table']}.{rel['to_column']}\n"
        
        summary += f"\n📐 MEASURES ({len(self.metadata['measures'])})\n"
        for name, measure in self.metadata["measures"].items():
            summary += f"  • {name}: {measure['expression']}\n"
        
        return summary

# Initialize metadata
metadata = SemanticModelMetadata(spark_loader)
print(metadata.get_summary())

## Section 4: Context Construction for AI
Build structured prompts for OpenAI that include complete model context for accurate DAX generation.

In [ ]:
class AIContextBuilder:
    """Build structured context for AI model (OpenAI)"""
    
    def __init__(self, metadata: SemanticModelMetadata):
        """Initialize context builder"""
        self.metadata = metadata
        self.base_prompt = self._build_base_prompt()
    
    def _build_base_prompt(self) -> str:
        """Build the base system prompt with model context"""
        prompt = """You are an expert Power BI DAX developer and Semantic Model specialist.
Your role is to understand Power BI Semantic Models and generate correct DAX expressions based on natural language requests.

IMPORTANT RULES:
1. Only use tables, columns, and relationships that exist in the model
2. Always validate that columns exist before using them
3. Use correct Power BI/DAX syntax
4. Follow Power BI naming conventions (PascalCase for measures, snake_case for columns)
5. Use relationships correctly - never create invalid joins
6. Avoid hallucinating columns or tables
7. Use aggregation functions when appropriate (SUM, COUNT, AVERAGE, etc.)
8. For time-based calculations, use the OrderDate column
9. For employee-related joins, use EmployeeKey or EmployeeID as appropriate
10. Generate clear, optimized DAX expressions

MODEL STRUCTURE:\n"""
        
        # Add tables and columns
        prompt += "\nTABLES AND COLUMNS:\n"
        for table_name, info in self.metadata.metadata["tables"].items():
            prompt += f"\n{table_name}:\n"
            for col_name, col_type in info["columns"].items():
                prompt += f"  - {col_name} ({col_type})\n"
        
        # Add relationships
        prompt += "\n\nRELATIONSHIPS:\n"
        for rel in self.metadata.metadata["relationships"]:
            prompt += f"• {rel['from_table']}.{rel['from_column']} → {rel['to_table']}.{rel['to_column']}\n"
        
        # Add existing measures
        prompt += "\n\nEXISTING MEASURES:\n"
        for name, measure in self.metadata.metadata["measures"].items():
            prompt += f"• {name}: {measure['expression']}\n"
            if measure.get('description'):
                prompt += f"  Description: {measure['description']}\n"
        
        prompt += "\n\nBUSINESS RULES:\n"
        prompt += "• Sales table is the fact table with granular transaction data\n"
        prompt += "• Product, Region, Salesperson are dimension tables\n"
        prompt += "• Use OrderDate for temporal analysis\n"
        prompt += "• EmployeeKey links Sales to Salesperson\n"
        prompt += "• RegionKey links Sales to Region\n"
        prompt += "• ProductKey links Sales to Product\n"
        
        return prompt
    
    def build_generation_prompt(self, user_request: str, item_type: str, 
                               conditions: str = "") -> str:
        """
        Build a focused prompt for DAX generation
        item_type: "measure", "flag", "column", etc.
        """
        prompt = self.base_prompt
        prompt += f"\n\nUSER REQUEST:\n"
        prompt += f"Type: {item_type}\n"
        prompt += f"Description: {user_request}\n"
        
        if conditions:
            prompt += f"Conditions: {conditions}\n"
        
        prompt += f"\n\nTASK:\nGenerate ONLY the DAX expression for a {item_type}.\n"
        prompt += "Response format should be:\n"
        prompt += "1. NAME: <suggested_name>\n"
        prompt += "2. EXPRESSION: <dax_code>\n"
        prompt += "3. EXPLANATION: <plain_english_explanation>\n"
        prompt += "4. VALIDATION: <any_issues_or_ok>\n"
        
        return prompt
    
    def build_explanation_prompt(self, dax_expression: str) -> str:
        """Build prompt for explaining DAX code"""
        prompt = f"""Explain this DAX expression in simple business language:

{dax_expression}

Provide:
1. Simple explanation of what it does
2. What tables/columns it uses
3. Any assumptions or constraints
4. Potential optimizations
"""
        return prompt
    
    def build_suggestion_prompt(self, dax_expression: str) -> str:
        """Build prompt for suggesting improvements"""
        prompt = f"""Review this DAX expression for a Power BI Semantic Model:

{dax_expression}

Using this model context:
{self.base_prompt}

Suggest:
1. Any performance optimizations
2. Best practice improvements
3. Naming convention suggestions
4. Alternative approaches if applicable
"""
        return prompt

# Initialize context builder
context_builder = AIContextBuilder(metadata)
logger.info("✓ AI Context Builder initialized")

## Section 5: DAX Generation Engine with OpenAI
Convert natural language queries to valid DAX expressions using gpt-4o-mini with prompt engineering.

In [ ]:
class DAXGenerationEngine:
    """Generate DAX expressions using OpenAI API"""
    
    def __init__(self, client, context_builder: AIContextBuilder):
        """Initialize DAX generation engine"""
        self.client = client
        self.context_builder = context_builder
        self.metadata = context_builder.metadata
        self.generation_history = []
    
    def generate_dax(self, user_request: str, item_type: str = "measure", 
                     conditions: str = "") -> Dict[str, Any]:
        """
        Generate DAX expression from natural language
        
        Args:
            user_request: What user wants to create
            item_type: Type of item (measure, flag, column, etc.)
            conditions: Any conditions for the item
        
        Returns:
            Dictionary with generated DAX, explanation, and validation
        """
        logger.info(f"Generating {item_type}: {user_request}")
        
        if not self.client:
            logger.warning("OpenAI client not available. Using fallback generation.")
            return self._fallback_generate(user_request, item_type, conditions)
        
        try:
            # Build focused prompt
            prompt = self.context_builder.build_generation_prompt(
                user_request, item_type, conditions
            )
            
            # Call OpenAI API
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a Power BI DAX expert."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,  # Low temperature for consistent, accurate DAX
                max_tokens=1000
            )
            
            result = response.choices[0].message.content
            parsed = self._parse_ai_response(result, item_type, user_request)
            
            # Store in history
            self.generation_history.append({
                "timestamp": datetime.now().isoformat(),
                "request": user_request,
                "type": item_type,
                "result": parsed
            })
            
            return parsed
            
        except Exception as e:
            logger.error(f"Error calling OpenAI API: {e}")
            return self._fallback_generate(user_request, item_type, conditions)
    
    def _parse_ai_response(self, response: str, item_type: str, 
                          request: str) -> Dict[str, Any]:
        """Parse OpenAI response into structured format"""
        result = {
            "type": item_type,
            "name": "",
            "expression": "",
            "explanation": "",
            "validation": "",
            "raw_response": response
        }
        
        # Extract name
        if "NAME:" in response:
            name_section = response.split("NAME:")[1].split("\n")[0].strip()
            result["name"] = name_section
        else:
            result["name"] = self._suggest_name(request, item_type)
        
        # Extract expression
        if "EXPRESSION:" in response:
            expr_section = response.split("EXPRESSION:")[1]
            if "EXPLANATION:" in expr_section:
                result["expression"] = expr_section.split("EXPLANATION:")[0].strip()
            else:
                result["expression"] = expr_section.strip()
        
        # Extract explanation
        if "EXPLANATION:" in response:
            expl_section = response.split("EXPLANATION:")[1]
            if "VALIDATION:" in expl_section:
                result["explanation"] = expl_section.split("VALIDATION:")[0].strip()
            else:
                result["explanation"] = expl_section.strip()
        
        # Extract validation
        if "VALIDATION:" in response:
            result["validation"] = response.split("VALIDATION:")[1].strip()
        
        return result
    
    def _suggest_name(self, request: str, item_type: str) -> str:
        """Suggest a name based on request"""
        # Remove common words
        words = request.lower().split()
        words = [w for w in words if w not in ["create", "add", "a", "the", "for", "where"]]
        
        # Create name
        if item_type == "flag":
            name = "_".join(words) + "_Flag"
        else:
            name = "_".join(words)
        
        # Clean up
        name = re.sub(r'[^a-zA-Z0-9_]', '', name)
        return name[:50]  # Limit length
    
    def _fallback_generate(self, user_request: str, item_type: str, 
                          conditions: str) -> Dict[str, Any]:
        """Fallback generation when API is not available"""
        logger.warning("Using fallback DAX generation (no OpenAI)")
        
        # Simple rule-based fallback
        request_lower = user_request.lower()
        
        result = {
            "type": item_type,
            "name": self._suggest_name(user_request, item_type),
            "expression": "",
            "explanation": "Generated using fallback mode (API not available)",
            "validation": "⚠️  Generated without AI validation",
            "raw_response": "Fallback mode"
        }
        
        # Simple pattern matching
        if "sales" in request_lower and "total" in request_lower:
            result["expression"] = "SUM(Sales[Sales])"
            result["explanation"] = "Sums all sales values from the Sales table"
        elif "count" in request_lower:
            result["expression"] = "COUNTA(Sales[OrderID])"
            result["explanation"] = "Counts total number of orders"
        elif "average" in request_lower or "avg" in request_lower:
            result["expression"] = "AVERAGE(Sales[Sales])"
            result["explanation"] = "Calculates average sales amount"
        else:
            result["expression"] = f"-- DAX for: {user_request}\n-- Please provide your OpenAI API key for AI generation"
            result["validation"] = "ERROR: Cannot generate without API key. Please set OPENAI_API_KEY environment variable."
        
        return result
    
    def generate_explanation(self, dax_expression: str) -> str:
        """Generate explanation for a DAX expression"""
        if not self.client:
            return "Explanation feature requires OpenAI API key"
        
        try:
            prompt = self.context_builder.build_explanation_prompt(dax_expression)
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                max_tokens=500
            )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error generating explanation: {e}")
            return f"Could not generate explanation: {str(e)}"
    
    def generate_suggestions(self, dax_expression: str) -> str:
        """Suggest improvements for a DAX expression"""
        if not self.client:
            return "Suggestions feature requires OpenAI API key"
        
        try:
            prompt = self.context_builder.build_suggestion_prompt(dax_expression)
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                max_tokens=800
            )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error generating suggestions: {e}")
            return f"Could not generate suggestions: {str(e)}"

# Initialize DAX engine
dax_engine = DAXGenerationEngine(openai_client, context_builder)
print("✓ DAX Generation Engine initialized")

## Section 6: Validation & Error Checking
Validate generated DAX for schema correctness, syntax, and logical consistency.

In [ ]:
class ValidationEngine:
    """Validate generated DAX expressions"""
    
    def __init__(self, metadata: SemanticModelMetadata):
        """Initialize validator"""
        self.metadata = metadata
        self.validation_rules = self._load_validation_rules()
    
    def _load_validation_rules(self) -> Dict:
        """Load DAX validation rules"""
        return {
            "required_functions": ["SUM", "COUNT", "AVERAGE", "MAX", "MIN", "DISTINCTCOUNT"],
            "forbidden_patterns": ["DELETE", "DROP", "INSERT", "CREATE TABLE"],
            "bracket_types": {"[": "]", "(": ")", "{": "}"}
        }
    
    def validate_dax(self, dax_expression: str) -> Tuple[bool, List[str]]:
        """
        Validate DAX expression
        
        Returns:
            (is_valid, list_of_errors)
        """
        errors = []
        
        # Basic syntax checks
        errors.extend(self._check_syntax(dax_expression))
        
        # Schema checks
        errors.extend(self._check_schema_references(dax_expression))
        
        # Security checks
        errors.extend(self._check_security(dax_expression))
        
        return len(errors) == 0, errors
    
    def _check_syntax(self, expression: str) -> List[str]:
        """Check basic DAX syntax"""
        errors = []
        
        # Check balanced brackets
        if not self._check_balanced_brackets(expression):
            errors.append("❌ Unbalanced brackets/parentheses in expression")
        
        # Check for empty expression
        if not expression or not expression.strip():
            errors.append("❌ Empty DAX expression")
        
        # Check for forbidden patterns
        for pattern in self.validation_rules["forbidden_patterns"]:
            if pattern in expression.upper():
                errors.append(f"❌ Forbidden operation: {pattern}")
        
        # Check for incomplete expressions
        if expression.strip().endswith(","):
            errors.append("❌ Expression ends with comma")
        
        return errors
    
    def _check_balanced_brackets(self, text: str) -> bool:
        """Check if brackets are balanced"""
        stack = []
        bracket_pairs = {"[": "]", "(": ")", "{": "}"}
        
        for char in text:
            if char in bracket_pairs:
                stack.append(char)
            elif char in bracket_pairs.values():
                if not stack:
                    return False
                if bracket_pairs[stack.pop()] != char:
                    return False
        
        return len(stack) == 0
    
    def _check_schema_references(self, expression: str) -> List[str]:
        """Check if tables and columns exist"""
        errors = []
        
        # Extract table[column] pattern
        pattern = r'(\w+)\[(\w+)\]'
        matches = re.findall(pattern, expression)
        
        for table_name, column_name in matches:
            # Check table exists
            if not self.metadata.check_table_exists(table_name):
                errors.append(f"⚠️  Table '{table_name}' not found in model")
            # Check column exists
            elif not self.metadata.check_column_exists(table_name, column_name):
                errors.append(f"⚠️  Column '{table_name}[{column_name}]' not found in model")
        
        return errors
    
    def _check_security(self, expression: str) -> List[str]:
        """Check for security issues"""
        errors = []
        
        # Check for SQL injection patterns
        dangerous_keywords = ["EXEC", "EXECUTE", "EVAL", "QUERY"]
        for keyword in dangerous_keywords:
            if keyword in expression.upper():
                errors.append(f"⚠️  Potentially dangerous keyword: {keyword}")
        
        return errors
    
    def validate_measure_name(self, name: str) -> Tuple[bool, str]:
        """Validate measure name"""
        if not name:
            return False, "Name cannot be empty"
        
        if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', name):
            return False, "Name must start with letter/underscore, contain only alphanumerics and underscores"
        
        if len(name) > 128:
            return False, "Name exceeds 128 character limit"
        
        if name.lower() in ["measure", "column", "table"]:
            return False, f"'{name}' is a reserved word"
        
        return True, "Name is valid"
    
    def suggest_fixes(self, errors: List[str]) -> List[str]:
        """Suggest fixes for validation errors"""
        suggestions = []
        
        for error in errors:
            if "Unbalanced brackets" in error:
                suggestions.append("→ Check that all ( have matching ) and [ have matching ]")
            elif "Table" in error and "not found" in error:
                suggestions.append("→ Use one of these tables: " + ", ".join(
                    self.metadata.metadata["tables"].keys()
                ))
            elif "Column" in error and "not found" in error:
                suggestions.append("→ Verify column name spelling and case sensitivity")
            elif "Expression ends with comma" in error:
                suggestions.append("→ Remove trailing comma from expression")
        
        return suggestions

# Initialize validator
validator = ValidationEngine(metadata)
print("✓ Validation Engine initialized")

## Section 7: Duplicate Detection & Registry
Maintain measure registry to prevent duplicates and suggest reuse of existing items.

In [ ]:
class MeasureRegistry:
    """Registry to track created measures and detect duplicates"""
    
    def __init__(self, metadata: SemanticModelMetadata):
        """Initialize registry"""
        self.metadata = metadata
        self.created_measures = {}
        self.creation_history = []
        
        # Load existing measures from metadata
        self._load_existing_measures()
    
    def _load_existing_measures(self):
        """Load measures from metadata"""
        for name, measure_info in self.metadata.metadata["measures"].items():
            self.created_measures[name] = {
                "expression": measure_info["expression"],
                "description": measure_info.get("description", ""),
                "created_at": None,  # Existing measures don't have timestamps
                "source": "existing"
            }
        logger.info(f"Loaded {len(self.created_measures)} existing measures")
    
    def measure_exists(self, measure_name: str) -> bool:
        """Check if a measure already exists"""
        return measure_name in self.created_measures
    
    def get_measure(self, measure_name: str) -> Optional[Dict]:
        """Get measure details"""
        return self.created_measures.get(measure_name)
    
    def find_similar_measures(self, description: str, threshold: float = 0.6) -> List[Tuple[str, float]]:
        """
        Find similar measures based on description similarity
        
        Returns:
            List of (measure_name, similarity_score)
        """
        similar = []
        
        # Simple similarity: count matching words
        request_words = set(description.lower().split())
        
        for measure_name, measure_info in self.created_measures.items():
            measure_words = set(
                (measure_info.get("description", "") + " " + measure_name).lower().split()
            )
            
            # Calculate Jaccard similarity
            if measure_words and request_words:
                intersection = len(request_words & measure_words)
                union = len(request_words | measure_words)
                similarity = intersection / union if union > 0 else 0
                
                if similarity >= threshold:
                    similar.append((measure_name, similarity))
        
        # Sort by similarity descending
        similar.sort(key=lambda x: x[1], reverse=True)
        return similar
    
    def register_measure(self, name: str, expression: str, description: str = "",
                        item_type: str = "measure") -> bool:
        """Register a newly created measure"""
        if self.measure_exists(name):
            logger.warning(f"Measure '{name}' already exists")
            return False
        
        self.created_measures[name] = {
            "expression": expression,
            "description": description,
            "created_at": datetime.now().isoformat(),
            "type": item_type,
            "source": "generated"
        }
        
        self.creation_history.append({
            "name": name,
            "type": item_type,
            "timestamp": datetime.now().isoformat(),
            "expression": expression
        })
        
        # Also add to metadata
        self.metadata.add_measure(name, expression, description)
        
        logger.info(f"✓ Registered new {item_type}: {name}")
        return True
    
    def get_all_measures(self) -> Dict[str, Dict]:
        """Get all registered measures"""
        return self.created_measures.copy()
    
    def get_registry_summary(self) -> str:
        """Get summary of all measures"""
        summary = "📋 MEASURE REGISTRY\n"
        summary += "=" * 60 + "\n\n"
        
        if not self.created_measures:
            summary += "No measures registered yet.\n"
            return summary
        
        # Separate by source
        existing = {k: v for k, v in self.created_measures.items() if v.get("source") == "existing"}
        generated = {k: v for k, v in self.created_measures.items() if v.get("source") == "generated"}
        
        if existing:
            summary += f"EXISTING MEASURES ({len(existing)}):\n"
            for name, info in existing.items():
                summary += f"  {name}:\n"
                summary += f"    Expression: {info['expression']}\n"
                if info.get('description'):
                    summary += f"    Description: {info['description']}\n"
                summary += "\n"
        
        if generated:
            summary += f"\nGENERATED MEASURES ({len(generated)}):\n"
            for name, info in generated.items():
                summary += f"  {name}:\n"
                summary += f"    Expression: {info['expression']}\n"
                if info.get('description'):
                    summary += f"    Description: {info['description']}\n"
                summary += f"    Created: {info.get('created_at', 'N/A')}\n"
                summary += "\n"
        
        return summary
    
    def export_registry(self, format: str = "json") -> str:
        """Export registry in specified format"""
        if format == "json":
            return json.dumps(self.created_measures, indent=2, default=str)
        elif format == "dax":
            # Export as DAX measure definitions
            output = "-- Generated Measures\n"
            for name, info in self.created_measures.items():
                if info.get("source") == "generated":
                    output += f"\n{name} =\n{info['expression']}\n"
            return output
        else:
            return str(self.created_measures)

# Initialize registry
measure_registry = MeasureRegistry(metadata)
print("✓ Measure Registry initialized")

## Section 8: Explanation & Documentation Module
Generate plain-English explanations of DAX and recommend optimizations.

In [ ]:
class ExplanationModule:
    """Generate explanations and documentation for DAX expressions"""
    
    def __init__(self, dax_engine: DAXGenerationEngine, metadata: SemanticModelMetadata):
        """Initialize explanation module"""
        self.dax_engine = dax_engine
        self.metadata = metadata
    
    def explain_dax(self, dax_expression: str, use_ai: bool = True) -> str:
        """
        Generate explanation for a DAX expression
        
        Args:
            dax_expression: The DAX code to explain
            use_ai: Whether to use AI (if available) or fallback
        """
        if use_ai:
            ai_explanation = self.dax_engine.generate_explanation(dax_expression)
            if "not available" not in ai_explanation.lower():
                return ai_explanation
        
        # Fallback to pattern-based explanation
        return self._pattern_based_explanation(dax_expression)
    
    def _pattern_based_explanation(self, expression: str) -> str:
        """Generate explanation using pattern matching"""
        explanation = "📝 EXPLANATION:\n"
        
        # Analyze the expression
        if "SUM(" in expression:
            explanation += "• This measure sums numerical values\n"
        
        if "COUNT(" in expression:
            explanation += "• This measure counts rows\n"
        
        if "AVERAGE(" in expression or "AVG(" in expression:
            explanation += "• This measure calculates an average\n"
        
        if "IF(" in expression:
            explanation += "• This measure includes conditional logic\n"
        
        if "FILTER(" in expression:
            explanation += "• This measure applies filters to data\n"
        
        if "CALCULATE(" in expression:
            explanation += "• This measure modifies the filter context\n"
        
        # Extract tables and columns
        pattern = r'(\w+)\[(\w+)\]'
        matches = re.findall(pattern, expression)
        
        if matches:
            explanation += "\n📊 TABLES & COLUMNS:\n"
            for table, column in set(matches):
                explanation += f"• {table}.{column}\n"
        
        explanation += "\n" + self._suggest_use_case(expression)
        
        return explanation
    
    def _suggest_use_case(self, expression: str) -> str:
        """Suggest use case for the measure"""
        suggestions = []
        expr_upper = expression.upper()
        
        if "SUM" in expr_upper and "SALES" in expr_upper:
            suggestions.append("Use for: Total revenue reporting and KPI dashboards")
        elif "COUNT" in expr_upper:
            suggestions.append("Use for: Volume analysis and transaction counting")
        elif "AVERAGE" in expr_upper:
            suggestions.append("Use for: Trend analysis and performance benchmarking")
        elif "IF" in expr_upper:
            suggestions.append("Use for: Conditional reporting and business rule implementation")
        else:
            suggestions.append("Use for: Custom business logic implementation")
        
        return "\n💡 " + suggestions[0] if suggestions else ""
    
    def generate_optimization_tips(self, dax_expression: str) -> List[str]:
        """Generate optimization recommendations"""
        tips = []
        
        # Check for performance issues
        if dax_expression.count("FILTER(") > 2:
            tips.append("⚡ Multiple FILTER functions detected - consider using CALCULATE instead")
        
        if "DISTINCTCOUNT" in dax_expression and "FILTER" in dax_expression:
            tips.append("⚡ DISTINCTCOUNT with FILTER can be slow - verify performance on large datasets")
        
        if len(dax_expression) > 500:
            tips.append("⚡ Expression is complex - consider breaking it into multiple measures")
        
        if dax_expression.count("(") != dax_expression.count(")"):
            tips.append("⚡ Check bracket balance - they appear to be unequal")
        
        if "%" not in dax_expression and "DIVIDE" not in dax_expression and "/" in dax_expression:
            tips.append("⚡ Consider using DIVIDE function to avoid division by zero errors")
        
        if not tips:
            tips.append("✅ Expression looks well-optimized!")
        
        return tips
    
    def create_documentation(self, measure_name: str, expression: str, 
                           description: str = "") -> str:
        """Create documentation for a measure"""
        doc = f"# {measure_name}\n\n"
        
        doc += f"## Description\n{description}\n\n" if description else ""
        
        doc += f"## Expression\n```dax\n{expression}\n```\n\n"
        
        doc += f"## Explanation\n{self._pattern_based_explanation(expression)}\n\n"
        
        doc += f"## Optimization Tips\n"
        for tip in self.generate_optimization_tips(expression):
            doc += f"- {tip}\n"
        
        return doc
    
    def generate_dax_script(self, measures: Dict[str, str]) -> str:
        """Generate a DAX script file with multiple measures"""
        script = "-- Power BI Semantic Model DAX Script\n"
        script += f"-- Generated: {datetime.now().isoformat()}\n"
        script += f"-- Total Measures: {len(measures)}\n\n"
        
        for measure_name, expression in measures.items():
            script += f"\n-- {measure_name}\n"
            script += f"{measure_name} =\n{expression}\n"
        
        return script

# Initialize explanation module
explanation_module = ExplanationModule(dax_engine, metadata)
print("✓ Explanation Module initialized")

## Section 9: Main Interactive Agent Loop
Multi-turn conversational agent that orchestrates all components for end-to-end DAX generation.

In [ ]:
class PowerBIAssistantAgent:
    """Main agent that orchestrates all components"""
    
    def __init__(self):
        """Initialize the agent with all components"""
        self.metadata = metadata
        self.registry = measure_registry
        self.engine = dax_engine
        self.validator = validator
        self.explainer = explanation_module
        self.session_items = []
        self.session_start = datetime.now()
        
        logger.info("✓ Power BI Assistant Agent initialized")
    
    def display_welcome(self):
        """Display welcome message"""
        print("\n" + "="*70)
        print("🤖 POWER BI SEMANTIC MODEL AI ASSISTANT")
        print("="*70)
        print("""
Welcome! I'm your AI-powered Power BI assistant. I understand your semantic model
and can generate correct DAX expressions from natural language.

WHAT CAN I DO?
• Create measures from descriptions
• Generate calculated columns with conditions
• Create flags with business logic
• Validate DAX syntax
• Explain existing expressions
• Suggest optimizations
• Prevent duplicate measures

EXAMPLES:
- "Create total sales measure"
- "Add high_performing_region_flag where sales > 100000"
- "Generate month-over-month growth calculation"

TYPE 'help' for commands, 'exit' to quit.
""")
        print("="*70 + "\n")
    
    def display_help(self):
        """Display help information"""
        help_text = """
AVAILABLE COMMANDS:
═══════════════════════════════════════════════════════════════

📊 CREATE ITEM
  'create [measure|flag|column] [description]'
  Example: create measure total sales
  
🔍 LIST REGISTRY
  'registry' or 'list' - Show all created measures
  
📋 CHECK MODEL
  'schema' or 'model' - Display semantic model structure
  
🧪 TEST DAX
  'validate [dax_expression]' - Validate DAX syntax
  Example: validate SUM(Sales[Sales])
  
📖 GET HELP
  'explain [dax]' - Explain a DAX expression
  'suggest [dax]' - Get optimization suggestions
  
📊 EXPORT
  'export [json|dax]' - Export registry
  'docs [measure_name]' - Generate documentation
  
ℹ️  OTHER
  'status' - Show session status
  'help' - Show this message
  'exit' or 'quit' - Exit the assistant

═══════════════════════════════════════════════════════════════
"""
        print(help_text)
    
    def process_user_input(self, user_input: str) -> bool:
        """
        Process user input and route to appropriate action
        Returns: True to continue, False to exit
        """
        user_input = user_input.strip()
        
        # Exit commands
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("\n👋 Thank you for using Power BI Assistant! Goodbye!\n")
            return False
        
        # Help
        if user_input.lower() in ["help", "?"]:
            self.display_help()
            return True
        
        # Status
        if user_input.lower() == "status":
            self._show_status()
            return True
        
        # Schema/Model
        if user_input.lower() in ["schema", "model", "structure"]:
            print(self.metadata.get_summary())
            return True
        
        # Registry
        if user_input.lower() in ["registry", "list", "measures"]:
            print(self.registry.get_registry_summary())
            return True
        
        # Export
        if user_input.lower().startswith("export"):
            parts = user_input.split()
            format_type = parts[1] if len(parts) > 1 else "json"
            self._handle_export(format_type)
            return True
        
        # Validate DAX
        if user_input.lower().startswith("validate"):
            dax_expr = user_input[9:].strip()
            self._handle_validation(dax_expr)
            return True
        
        # Explain DAX
        if user_input.lower().startswith("explain"):
            dax_expr = user_input[8:].strip()
            self._handle_explanation(dax_expr)
            return True
        
        # Suggest improvements
        if user_input.lower().startswith("suggest"):
            dax_expr = user_input[8:].strip()
            self._handle_suggestions(dax_expr)
            return True
        
        # Generate documentation
        if user_input.lower().startswith("docs"):
            measure_name = user_input[5:].strip()
            self._handle_documentation(measure_name)
            return True
        
        # Create item
        if user_input.lower().startswith("create"):
            self._handle_creation(user_input)
            return True
        
        # Default: treat as creation request
        if any(keyword in user_input.lower() for keyword in ["measure", "flag", "column"]):
            self._handle_creation_interactive(user_input)
            return True
        
        # Unknown command
        print(f"❓ Command not recognized: '{user_input}'")
        print("Type 'help' for available commands.\n")
        return True
    
    def _handle_creation(self, user_input: str):
        """Handle item creation"""
        parts = user_input.split(maxsplit=1)
        
        if len(parts) < 2:
            print("Usage: create [measure|flag|column] [description]")
            return
        
        item_type = "measure"  # Default
        description = parts[1]
        
        # Extract type if specified
        if "measure" in description.lower():
            item_type = "measure"
            description = description.lower().replace("measure", "").strip()
        elif "flag" in description.lower():
            item_type = "flag"
            description = description.lower().replace("flag", "").strip()
        elif "column" in description.lower():
            item_type = "column"
            description = description.lower().replace("column", "").strip()
        
        # Ask for conditions if needed
        conditions = ""
        if item_type in ["flag", "column"]:
            conditions = input(f"  Enter conditions (or press Enter to skip): ")
        
        self._execute_generation(description, item_type, conditions)
    
    def _handle_creation_interactive(self, hint: str = ""):
        """Interactive creation flow"""
        print("\n" + "="*70)
        print("✨ CREATE NEW ITEM")
        print("="*70)
        
        # Get item type
        print("\\nWhat do you want to create?")
        print("  1) Measure (aggregation or calculation)")
        print("  2) Flag (conditional field)")
        print("  3) Calculated Column")
        choice = input("\\nEnter choice (1-3): ").strip()
        
        item_types = {"1": "measure", "2": "flag", "3": "column"}
        item_type = item_types.get(choice, "measure")
        
        # Get description
        print(f"\\nDescribe what the {item_type} should do:")
        description = input("> ").strip()
        
        # Get conditions if applicable
        conditions = ""
        if item_type in ["flag", "column"]:
            print(f"\\nEnter any conditions for the {item_type} (optional):")
            conditions = input("> ").strip()
        
        self._execute_generation(description, item_type, conditions)
    
    def _execute_generation(self, description: str, item_type: str, conditions: str = ""):
        """Execute DAX generation pipeline"""
        print(f"\\n⏳ Generating {item_type}...")
        print("="*70)
        
        # Step 1: Check for similar measures
        print("\\n[1/6] Checking for similar measures...", end=" ")
        similar = self.registry.find_similar_measures(description)
        if similar:
            print(f"✓ Found {len(similar)} similar:")
            for name, score in similar[:3]:
                measure = self.registry.get_measure(name)
                print(f"      • {name} (Match: {score:.0%})")
                print(f"        {measure['expression']}")
            
            reuse = input("\\n      Reuse one of these? (name or 'n' for new): ").strip().lower()
            if reuse in [n for n, _ in similar]:
                print(f"\\n✓ Using existing measure: {reuse}")
                return
        else:
            print("✓ No similar measures found")
        
        # Step 2: Generate DAX
        print("[2/6] Generating DAX with AI...", end=" ")
        result = self.engine.generate_dax(description, item_type, conditions)
        print("✓")
        
        # Step 3: Validate
        print("[3/6] Validating syntax...", end=" ")
        is_valid, errors = self.validator.validate_dax(result["expression"])
        if is_valid:
            print("✓")
        else:
            print("⚠️")
            for error in errors:
                print(f"      {error}")
        
        # Step 4: Check duplicates
        print("[4/6] Checking for duplicates...", end=" ")
        if self.registry.measure_exists(result["name"]):
            print(f"⚠️ Measure '{result['name']}' already exists!")
            existing = self.registry.get_measure(result["name"])
            print(f"      Current: {existing['expression']}")
            return
        else:
            print("✓")
        
        # Step 5: Explain
        print("[5/6] Generating explanation...", end=" ")
        explanation = self.explainer.explain_dax(result["expression"])
        print("✓")
        
        # Step 6: Show results
        print("[6/6] Finalizing...", end=" ")
        print("✓")
        
        # Display results
        print("\\n" + "="*70)
        print("✨ GENERATION COMPLETE")
        print("="*70)
        
        print(f"\\n📊 {item_type.upper()}: {result['name']}")
        print(f"\\n  Expression:\\n  {result['expression']}")
        print(f"\\n  Explanation:\\n  {explanation}")
        
        if not is_valid and errors:
            print(f"\\n  ⚠️  Validation Issues:")
            for error in errors:
                print(f"     {error}")
        
        print(f"\\n  Optimization Tips:")
        for tip in self.explainer.generate_optimization_tips(result["expression"]):
            print(f"  {tip}")
        
        # Ask to save
        print("\\n" + "="*70)
        save = input("\\nSave to registry? (y/n): ").strip().lower()
        if save == "y":
            self.registry.register_measure(
                result["name"],
                result["expression"],
                description,
                item_type
            )
            print(f"\\n✓ {item_type.capitalize()} '{result['name']}' saved!")
            self.session_items.append(result["name"])
        else:
            print("\\n✗ Not saved (can be recreated later)")
        
        print()
    
    def _handle_validation(self, dax_expr: str):
        """Handle DAX validation"""
        print(f"\\n🧪 Validating: {dax_expr[:50]}...")
        is_valid, errors = self.validator.validate_dax(dax_expr)
        
        if is_valid:
            print("\\n✅ Expression is valid!")
        else:
            print("\\n❌ Issues found:")
            for error in errors:
                print(f"  {error}")
            
            suggestions = self.validator.suggest_fixes(errors)
            if suggestions:
                print("\\n💡 Suggestions:")
                for sugg in suggestions:
                    print(f"  {sugg}")
        print()
    
    def _handle_explanation(self, dax_expr: str):
        """Handle DAX explanation"""
        print(f"\\n📖 Explaining DAX...")
        explanation = self.explainer.explain_dax(dax_expr)
        print(f"\\n{explanation}")
        print()
    
    def _handle_suggestions(self, dax_expr: str):
        """Handle optimization suggestions"""
        print(f"\\n💡 Analyzing for optimizations...")
        suggestions = self.explainer.generate_optimization_tips(dax_expr)
        print("\\nOptimization Tips:")
        for tip in suggestions:
            print(f"  {tip}")
        print()
    
    def _handle_documentation(self, measure_name: str):
        """Generate documentation for a measure"""
        measure = self.registry.get_measure(measure_name)
        if not measure:
            print(f"\\n❌ Measure '{measure_name}' not found")
            return
        
        doc = self.explainer.create_documentation(
            measure_name,
            measure["expression"],
            measure.get("description", "")
        )
        print(f"\\n{doc}")
    
    def _handle_export(self, format_type: str):
        """Handle registry export"""
        exported = self.registry.export_registry(format_type)
        print(f"\\n📤 Exporting in {format_type.upper()} format...")
        print(f"\\n{exported}\\n")
    
    def _show_status(self):
        """Show session status"""
        elapsed = datetime.now() - self.session_start
        print(f"\\n📊 SESSION STATUS")
        print(f"  Time: {elapsed}")
        print(f"  Items created: {len(self.session_items)}")
        print(f"  Total measures: {len(self.registry.created_measures)}")
        if self.session_items:
            print(f"  This session: {', '.join(self.session_items)}")
        print()
    
    def run_interactive_loop(self):
        """Main interactive loop"""
        self.display_welcome()
        
        while True:
            try:
                user_input = input("\\n🤖 You> ").strip()
                
                if not user_input:
                    continue
                
                if not self.process_user_input(user_input):
                    break
                    
            except KeyboardInterrupt:
                print("\\n\\n👋 Session interrupted. Goodbye!")
                break
            except Exception as e:
                logger.error(f"Unexpected error: {e}")
                print(f"\\n❌ Error: {str(e)}")
                print("Please try again or type 'help' for commands.\\n")

# Initialize the agent
pbi_assistant = PowerBIAssistantAgent()
logger.info("✓ Power BI Assistant Agent ready to use")

## Quick Start Guide

Run the cell below to launch the interactive Power BI Assistant!

In [ ]:
# LAUNCH THE INTERACTIVE ASSISTANT
# =====================================
# Uncomment the line below to start the interactive session
# Note: This will accept user input and run in interactive mode

# pbi_assistant.run_interactive_loop()

# For now, let's demonstrate with programmatic examples (see next cells)

## Example 1: Check Current Model Structure

In [ ]:
print(metadata.get_summary())

## Example 2: Generate a Simple Measure

In [ ]:
# Generate a measure for total revenue
request = "Create a total revenue measure that sums all sales"
print(f"📊 Request: {request}\n")

result = dax_engine.generate_dax(request, "measure")

print(f"✨ Generated Measure: {result['name']}")
print(f"   Expression: {result['expression']}")
print(f"   Explanation: {result['explanation']}")
print(f"   Validation: {result['validation']}")

# Validate the generated DAX
is_valid, errors = validator.validate_dax(result['expression'])
print(f"\n✅ Validation: {'PASSED' if is_valid else 'FAILED'}")
if errors:
    for error in errors:
        print(f"   {error}")

## Example 3: Create a Flag with Conditions

In [ ]:
# Generate a flag for high-performing sales
request = "Add high performing sales flag"
conditions = "Where total sales is greater than 50000"

print(f"🚩 Request: {request}")
print(f"   Conditions: {conditions}\n")

result = dax_engine.generate_dax(request, "flag", conditions)

print(f"✨ Generated Flag: {result['name']}")
print(f"   Expression: {result['expression']}")
print(f"   Explanation: {result['explanation']}")

# Check if this already exists
is_duplicate = measure_registry.measure_exists(result['name'])
print(f"\n   Duplicate Check: {'DUPLICATE ⚠️' if is_duplicate else 'NEW ✓'}")

## Example 4: Validate and Explain Existing DAX

In [ ]:
# Test validation and explanation on existing DAX
test_dax = "SUM(Sales[Sales]) / COUNTA(Sales[OrderID])"

print(f"📋 Testing DAX: {test_dax}\n")

# Validate
is_valid, errors = validator.validate_dax(test_dax)
print(f"✅ Validation: {'PASSED' if is_valid else 'FAILED'}")
if errors:
    for error in errors:
        print(f"   {error}")
else:
    print("   All checks passed!")

# Explain
print(f"\n📖 Explanation:")
explanation = explanation_module.explain_dax(test_dax)
print(explanation)

# Get optimization tips
print(f"\n💡 Optimization Tips:")
tips = explanation_module.generate_optimization_tips(test_dax)
for tip in tips:
    print(f"   {tip}")

## Example 5: View and Manage Measure Registry

In [ ]:
# View current measure registry
print(measure_registry.get_registry_summary())

# Register a new measure
print("\n" + "="*70)
print("Registering a new measure...")
print("="*70 + "\n")

measure_registry.register_measure(
    "Average_Transaction_Value",
    "SUM(Sales[Sales]) / COUNTA(Sales[OrderID])",
    "Average value of each sales transaction",
    "measure"
)

# Show updated registry
print("\n📋 Updated Registry:\n")
print(measure_registry.get_registry_summary())

# Export as DAX script
print("\n📤 Export as DAX Script:\n")
print(measure_registry.export_registry("dax"))

## Advanced: Full Interactive Session Example

To start the full interactive assistant with multi-turn conversations, run this cell: